# Natural frequency analysis of a plate

A simply supported plate of `Quad4` elements, whose lowest natural frequency
is compared with the exact one of Leissa (1969), with the matrices of
`plate_matrices` of `tests/test_solver.py`, see
[Linear buckling analysis of a plate](ex_linear_buckling_plate.ipynb). The
mass matrix is assembled with `update_M` and the frequencies are computed
with `natural_frequency`, with and without the equilibration of the
diagonal. The analysis is described in
[Linear buckling and natural frequencies](../eigenvalue_analyses.rst) and the
mass matrices in [Mass matrices of the shells](../shell_mass.rst); the full
references are at the end of the notebook.

## The reference value

The free vibrations $\boldsymbol{u} \sin(\omega_n t)$ give
$(\boldsymbol{K}_{C0} - \omega_n^2 \boldsymbol{M}) \boldsymbol{u} =
\boldsymbol{0}$. For the Kirchhoff plate with all edges simply supported,
Leissa (1969, Sec. 4.1, p. 44, Eq. (4.20)) gives the exact frequencies

$$
\omega_{mn} = \sqrt{\frac{D}{\rho h}} \left[\left(\frac{m \pi}{a}\right)^2
+ \left(\frac{n \pi}{b}\right)^2\right], \qquad
D = \frac{E h^3}{12 (1 - \nu^2)}
$$

with $\rho h$ the mass per unit area (Leissa's $\rho$); the lowest is $m = n =
1$. The plate of `plate_matrices` has $a = 2$ m, $b = 0.5$ m, $h = 0.002$ m,
$E = 203$ GPa, $\nu = 0.33$ and $\rho = 7830$ kg/m$^3$, meshed here with 60 x
18 elements and the consistent mass matrix. With $b/h = 250$ the frequency of
the Mindlin plate, with its rotary inertia, is 0.003 % below the Kirchhoff
value. The test compares $\omega_{11}$ with `rtol = 0.01`, and the
frequencies with and without the equilibration with `rtol = 1e-6`:

In [1]:
import os
import sys

sys.path.insert(0, os.path.join('..', 'tests'))

import numpy as np
import pytest
from scipy.sparse import coo_matrix
from pyfe3d import DOF, DOUBLE, INT, Quad4, Quad4Data, Quad4Probe
from pyfe3d.shellprop_utils import isotropic_plate
from pyfe3d.solver import natural_frequency
from test_solver import (A_PLATE, B_PLATE, E_PLATE, H_PLATE, NU_PLATE,
                         RHO_PLATE, plate_matrices)

In [2]:
@pytest.mark.slow
def test_natural_frequency_plate():
    r"""End-to-end against the natural frequency of a simply supported
    plate of Leissa (1969), Eq. (4.20), see the module documentation

    With ``b/h = 250`` the frequency of the Mindlin plate, with its rotary
    inertia, is 0.003 % below the reference. With 60x18 elements and the
    consistent mass matrix the frequency is 0.32 % above it, with 20x6
    elements 3.05 %, with 40x12 elements 0.74 %, with 80x24 elements
    0.17 %.
    """
    KC0uu, KGuu, Muu = plate_matrices(with_mass=True, nx=61, ny=19)
    omegan, eigvecs = natural_frequency(KC0uu, Muu, num_eigvalues=4)
    D = E_PLATE*H_PLATE**3/(12*(1 - NU_PLATE**2))
    wmn = np.sqrt(D/(RHO_PLATE*H_PLATE))*((np.pi/A_PLATE)**2
                                          + (np.pi/B_PLATE)**2)
    print('omegan[0] %.4f  theory %.4f' % (omegan[0], wmn))
    assert np.isclose(wmn, omegan[0], rtol=0.01)
    assert np.all(np.diff(omegan) >= -1e-9)

    # NOTE same frequencies without the scaling
    omegan_raw, _ = natural_frequency(KC0uu, Muu, num_eigvalues=4,
                                      precondition=False)
    print('preconditioned', omegan[:3], 'raw', omegan_raw[:3])
    assert np.allclose(omegan, omegan_raw, rtol=1e-6)

In [3]:
test_natural_frequency_plate()

omegan[0] 131.0857  theory 130.6267


preconditioned [131.08565561 154.14808495 192.69001222] raw [131.08565561 154.14808495 192.69001222]


The printed frequency is 0.35 % above the reference, and the equilibration
of the diagonal changes none of the three lowest frequencies.

## The three mass matrices

The consistent, lumped and reduced-integration mass matrices of `Quad4`,
`mtype` 0, 1 and 2 of `update_M`, on the plate $a = 0.3$ m, $b = 0.5$ m, $h =
0.01$ m, $E = 203$ GPa, $\nu = 0.33$, $\rho = 7830$ kg/m$^3$
(`tests/test_quad4_natural_freq.py`). The reference is Eq. (4.20) of Leissa
(1969) above, with `rtol = 0.01` for each `mtype`. The edges are hard simply
supported, $w = 0$ and the rotation about the axis of the edge null, for which
the Navier mode is also exact for the Mindlin plate: with $a/h = 30$, the
rotary inertia and $A_{44} = A_{55} = (5/6) G h$, its frequency is 0.28 %
below the Kirchhoff value, from the characteristic equation (76) of Wittrick
(1987, Sec. 3.2, p. 457), and the elements converge to that value. With the
soft simple support, rotations free, they would converge to about 1 % below
the reference, through the boundary layer of the Mindlin plate along the
edges.

In [4]:
@pytest.mark.slow
def test_nat_freq_plate(plot=False, mode=0, mtypes=range(3), refinement=3):
    r"""Fundamental frequency of a simply supported plate, ``a = 0.3``,
    ``b = 0.5``, ``h = 0.01``, with the consistent, lumped and reduced mass
    matrices

    The reference is the exact solution of Leissa, A. W., Vibration of
    Plates, NASA SP-160, 1969, Sec. 4.1, p. 44, Eq. (4.20),
    ``omega = sqrt(D/rho)*((m*pi/a)**2 + (n*pi/b)**2)``, with ``m = n = 1``
    and Leissa's ``rho`` the mass per unit area, here ``rho*h``.

    The reference is that of the Kirchhoff plate. The edges are hard simply
    supported, ``w = 0`` and the rotation about the axis of the edge equal to
    zero, for which the Navier mode is also exact for the Mindlin plate: with
    ``a/h = 30``, the rotary inertia and ``A44 = A55 = 5/6 G h``, its
    frequency is 0.28 % below the reference, from the characteristic equation
    (76) of Wittrick, W. H., Analytical, three-dimensional elasticity
    solutions to some plate problems, and some observations on Mindlin's
    plate theory, International Journal of Solids and Structures, 23(4),
    441-464, 1987, Sec. 3.2, p. 457, and the elements converge to that
    value. With the soft simple support used before, rotations free, they
    converge to about 1 % below the reference, through the boundary layer
    of the Mindlin plate along the edges.

    With 26x32 elements the frequencies of the three mass matrices are
    0.03 to 0.33 % below the reference, with 17x21 elements from 0.40 %
    below to 0.31 % above it, with 8x10 elements from 0.78 % below to 2.42 %
    above it, with 35x43 elements 0.14 to 0.31 % below it.
    """
    data = Quad4Data()
    probe = Quad4Probe()
    for mtype in mtypes:
        nx = refinement*9
        ny = refinement*11

        a = 0.3
        b = 0.5

        E = 203.e9 # Pa
        nu = 0.33

        rho = 7.83e3 # kg/m3
        h = 0.01 # m

        xtmp = np.linspace(0, a, nx)
        ytmp = np.linspace(0, b, ny)
        xmesh, ymesh = np.meshgrid(xtmp, ytmp)
        ncoords = np.vstack((xmesh.T.flatten(), ymesh.T.flatten(), np.zeros_like(ymesh.T.flatten()))).T

        x = ncoords[:, 0]
        y = ncoords[:, 1]
        z = ncoords[:, 2]
        ncoords_flatten = ncoords.flatten()

        nids = 1 + np.arange(ncoords.shape[0])
        nid_pos = dict(zip(nids, np.arange(len(nids))))
        nids_mesh = nids.reshape(nx, ny)
        n1s = nids_mesh[:-1, :-1].flatten()
        n2s = nids_mesh[1:, :-1].flatten()
        n3s = nids_mesh[1:, 1:].flatten()
        n4s = nids_mesh[:-1, 1:].flatten()

        num_elements = len(n1s)
        print('num_elements', num_elements)

        KC0r = np.zeros(data.KC0_SPARSE_SIZE*num_elements, dtype=INT)
        KC0c = np.zeros(data.KC0_SPARSE_SIZE*num_elements, dtype=INT)
        KC0v = np.zeros(data.KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
        Mr = np.zeros(data.M_SPARSE_SIZE*num_elements, dtype=INT)
        Mc = np.zeros(data.M_SPARSE_SIZE*num_elements, dtype=INT)
        Mv = np.zeros(data.M_SPARSE_SIZE*num_elements, dtype=DOUBLE)
        N = DOF*nx*ny

        prop = isotropic_plate(thickness=h, E=E, nu=nu, rho=rho)

        quads = []
        init_k_KC0 = 0
        init_k_M = 0
        for n1, n2, n3, n4 in zip(n1s, n2s, n3s, n4s):
            pos1 = nid_pos[n1]
            pos2 = nid_pos[n2]
            pos3 = nid_pos[n3]
            pos4 = nid_pos[n4]
            r1 = ncoords[pos1]
            r2 = ncoords[pos2]
            r3 = ncoords[pos3]
            normal = np.cross(r2 - r1, r3 - r2)[2]
            assert normal > 0
            quad = Quad4(probe)
            quad.n1 = n1
            quad.n2 = n2
            quad.n3 = n3
            quad.n4 = n4
            quad.c1 = DOF*nid_pos[n1]
            quad.c2 = DOF*nid_pos[n2]
            quad.c3 = DOF*nid_pos[n3]
            quad.c4 = DOF*nid_pos[n4]
            quad.init_k_KC0 = init_k_KC0
            quad.init_k_M = init_k_M
            quad.K6ROT = 100.
            quad.update_rotation_matrix(ncoords_flatten)
            quad.update_probe_xe(ncoords_flatten)
            quad.update_KC0(KC0r, KC0c, KC0v, prop)
            quad.update_M(Mr, Mc, Mv, prop, mtype=mtype)
            quads.append(quad)
            init_k_KC0 += data.KC0_SPARSE_SIZE
            init_k_M += data.M_SPARSE_SIZE

        print('elements created')

        KC0 = coo_matrix((KC0v, (KC0r, KC0c)), shape=(N, N)).tocsc()
        M = coo_matrix((Mv, (Mr, Mc)), shape=(N, N)).tocsc()

        print('sparse KC0 and M created')

        bk = np.zeros(N, dtype=bool)
        check = np.isclose(x, 0.) | np.isclose(x, a) | np.isclose(y, 0) | np.isclose(y, b)
        bk[0::DOF] = check
        bk[1::DOF] = check
        bk[2::DOF] = check
        # NOTE hard simple support: rotation about the x axis on the edges
        #      x = 0, a and about the y axis on the edges y = 0, b
        bk[3::DOF] = np.isclose(x, 0.) | np.isclose(x, a)
        bk[4::DOF] = np.isclose(y, 0.) | np.isclose(y, b)

        bu = ~bk

        KC0uu = KC0[bu, :][:, bu]
        Muu = M[bu, :][:, bu]

        num_eigenvalues = max(2, mode+1)

        print('eig solver begin')
        # solves Ax = lambda M x
        # we have Ax - lambda M x = 0, with lambda = omegan**2

        # NOTE pyfe3d.solver.natural_frequency equilibrates the diagonal
        #      before calling the eigensolver and returns the circular
        #      frequencies already sorted
        omegan, eigvecsu = natural_frequency(KC0uu, Muu, num_eigvalues=num_eigenvalues, tol=1e-6)

        print('eig solver end')

        eigvecs = np.zeros((N, eigvecsu.shape[1]), dtype=float)
        eigvecs[bu, :] = eigvecsu

        u = np.zeros(N)
        u[bu] = eigvecsu[:, mode]

        # NOTE Leissa (1969), Eq. (4.20), with rho*h the mass per unit area
        m = 1
        n = 1
        D = E*h**3/(12*(1 - nu**2))
        wmn = np.sqrt(D/(rho*h))*((m*np.pi/a)**2 + (n*np.pi/b)**2)

        print('Theoretical omega123', wmn)
        print('Numerical omega123', omegan[0:10])
        assert np.isclose(wmn, omegan[0], rtol=0.01)

    if plot:
        import matplotlib.pyplot as plt

        plt.clf()
        plt.contourf(xmesh, ymesh, u[2::DOF].reshape(nx, ny).T)
        plt.show()

In [5]:
test_nat_freq_plate()

num_elements 832


elements created
sparse KC0 and M created
eig solver begin


eig solver end
Theoretical omega123 2322.2524909217354
Numerical omega123 [2319.20901195 4158.67263866]
num_elements 832


elements created
sparse KC0 and M created
eig solver begin


eig solver end
Theoretical omega123 2322.2524909217354
Numerical omega123 [2314.52926907 4140.29286804]
num_elements 832


elements created
sparse KC0 and M created
eig solver begin


eig solver end
Theoretical omega123 2322.2524909217354
Numerical omega123 [2321.55613649 4167.93013728]


With the default `refinement=3`, 26 x 32 elements, the fundamental frequency
is 0.13 % (consistent), 0.33 % (lumped) and 0.03 % (reduced integration)
below the reference.

## References

- Leissa, A. W., Vibration of Plates, NASA SP-160, 1969.
- Wittrick, W. H., Analytical, three-dimensional elasticity solutions to some
  plate problems, and some observations on Mindlin's plate theory,
  International Journal of Solids and Structures, 23(4), 441-464, 1987.